# 03. LLM-анализ HDBSCAN-кластеров

Этот ноутбук читает результат HDBSCAN из `hdbscan_clusters.parquet`, берет несколько характерных примеров из каждого кластера и с помощью LLM описывает особенность каждого кластера.

Ноутбук **не пересчитывает HDBSCAN** и **не размечает каждый отзыв отдельно**. Он анализирует уже найденные кластеры.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Если groq не установлен, раскомментируй и выполни:
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.5 MB/s eta 0:00:00


In [3]:
import os
import json
import time
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm.auto import tqdm
from groq import Groq

## 1. Настройки

Проверь `BASE_DIR` и `RESULT_DIR`. Они должны совпадать с путями во втором ноутбуке, где сохранялись HDBSCAN-кластеры.

In [4]:
# =========================
# Пути проекта
# =========================

BASE_DIR = Path("/content/drive/MyDrive/MLops_project/project")
DATA_DIR = BASE_DIR / "data"

# Директория с результатами второго HDBSCAN-ноутбука
RESULT_DIR = DATA_DIR / "hdbscan_results_raw_sample_200k"

# Входной файл с результатом HDBSCAN
CLUSTERS_PATH = RESULT_DIR / "hdbscan_clusters.parquet"

# Выходные файлы анализа кластеров
CLUSTER_ANALYSIS_CSV_PATH = RESULT_DIR / "cluster_llm_analysis.csv"
CLUSTER_ANALYSIS_PARQUET_PATH = RESULT_DIR / "cluster_llm_analysis.parquet"
CLUSTER_ANALYSIS_JSONL_PATH = RESULT_DIR / "cluster_llm_analysis.jsonl"

# Объединенный результат: каждый отзыв + описание его кластера
ENRICHED_CLUSTERS_PATH = RESULT_DIR / "hdbscan_clusters_with_llm_analysis.parquet"
ENRICHED_CLUSTERS_CSV_PATH = RESULT_DIR / "hdbscan_clusters_with_llm_analysis.csv"

# =========================
# LLM API как в data_markup
# =========================

API_KEY_ENV_NAME = "GROQ_API_KEY"
MODEL_NAME = "qwen/qwen3-32b"

TEMPERATURE = 0
MAX_RETRIES = 3
RETRY_BASE_SLEEP_SECONDS = 2

# Пауза между запросами. Если API начнет ругаться на лимиты, поставь 0.5 / 1 / 2 / 5.
REQUEST_SLEEP_SECONDS = 0.2

# Если True, старый анализ будет удален и начат заново.
# Если False, ноутбук продолжит с места остановки.
RESET_CLUSTER_ANALYSIS = False

In [5]:
# =========================
# Параметры анализа кластеров
# =========================

# Анализировать ли шумовой кластер -1.
# Обычно лучше False, потому что -1 — это шум, а не единая тема.
ANALYZE_NOISE_CLUSTER = False

# Минимальный размер кластера для анализа.
# Маленькие кластеры можно пропустить, чтобы не тратить API.
MIN_CLUSTER_SIZE_TO_ANALYZE = 50

# Сколько примеров передавать LLM на один кластер.
N_TOP_EXAMPLES = 25

# Брать самые уверенные примеры по cluster_probability.
USE_TOP_PROBABILITY_EXAMPLES = True

# Максимальная длина одного отзыва в prompt.
MAX_REVIEW_CHARS = 500

# Ограничение количества кластеров для теста.
# Например, 20 — проанализировать только 20 крупнейших кластеров.
# None — анализировать все подходящие кластеры.
MAX_CLUSTERS_TO_ANALYZE = None

In [6]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if API_KEY_ENV_NAME not in os.environ or not os.environ[API_KEY_ENV_NAME]:
    os.environ[API_KEY_ENV_NAME] = getpass(f"Вставь {API_KEY_ENV_NAME}: ")

client = Groq(api_key=os.environ[API_KEY_ENV_NAME])

print("MODEL_NAME:", MODEL_NAME)
print("RESULT_DIR:", RESULT_DIR)

Вставь GROQ_API_KEY: ··········
MODEL_NAME: qwen/qwen3-32b
RESULT_DIR: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k


## 2. Загрузка HDBSCAN-кластеров

In [7]:
if not CLUSTERS_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл с кластерами: {CLUSTERS_PATH}\n"
        "Сначала запусти второй ноутбук с HDBSCAN, который создает hdbscan_clusters.parquet."
    )

result_df = pd.read_parquet(CLUSTERS_PATH)

print("Размер result_df:", result_df.shape)
print("Колонки:", list(result_df.columns))

display(result_df.head())


Размер result_df: (178132, 12)
Колонки: ['nmId', 'productValuation', 'color', 'text', 'answer', 'source_row_idx', 'source_file', 'source_text_col', 'sample_part_id', 'sample_file', 'cluster_id', 'cluster_probability']


,nmId,productValuation,color,text,answer,source_row_idx,source_file,source_text_col,sample_part_id,sample_file,cluster_id,cluster_probability
0,0,5,темно-серый,качество хорошее. цвет темно серый. Очень прог...,,75721,raw_part_00000.parquet,text,0,sample_part_00000.parquet,-1,0.000000
1,0,5,черный,Классная лампа. Не хлипкая. Очень удобная. Бра...,Добрый день! благодарим за выбор нашей продукц...,80184,raw_part_00000.parquet,text,0,sample_part_00000.parquet,222,1.000000
2,0,4,черный,На узкую ногу,Здравствуйте! Спасибо за отзыв. Мы очень сожал...,19864,raw_part_00000.parquet,text,0,sample_part_00000.parquet,238,0.862404
3,0,5,темно-синий,"Штаны хорошие, обычные брюки цена в 1300 в сам...",,76699,raw_part_00000.parquet,text,0,sample_part_00000.parquet,121,0.421791
4,10002976,5,красный,"Думали порвутся сразу, но как подарок на нг со...",Добрый день!\n Мы очень признательны вам за п...,92991,raw_part_00000.parquet,text,0,sample_part_00000.parquet,-1,0.000000


In [8]:
required_cols = ["text", "cluster_id", "cluster_probability"]
missing_cols = [col for col in required_cols if col not in result_df.columns]

if missing_cols:
    raise ValueError(f"Не хватает колонок: {missing_cols}")

result_df = result_df.copy()
result_df["text"] = result_df["text"].astype(str)
result_df["cluster_id"] = result_df["cluster_id"].astype(int)
result_df["cluster_probability"] = result_df["cluster_probability"].astype(float)

print("Количество строк:", len(result_df))
print("Количество cluster_id, включая -1:", result_df["cluster_id"].nunique())
print("Доля шума cluster_id=-1:", (result_df["cluster_id"] == -1).mean())

Количество строк: 178132
Количество cluster_id, включая -1: 298
Доля шума cluster_id=-1: 0.5083926526396155


## 3. Подготовка списка кластеров для LLM

In [9]:
cluster_summary_for_llm = (
    result_df
    .groupby("cluster_id")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

if not ANALYZE_NOISE_CLUSTER:
    cluster_summary_for_llm = cluster_summary_for_llm[
        cluster_summary_for_llm["cluster_id"] != -1
    ]

cluster_summary_for_llm = cluster_summary_for_llm[
    cluster_summary_for_llm["count"] >= MIN_CLUSTER_SIZE_TO_ANALYZE
].copy()

cluster_summary_for_llm["share"] = cluster_summary_for_llm["count"] / len(result_df)

if MAX_CLUSTERS_TO_ANALYZE is not None:
    cluster_summary_for_llm = cluster_summary_for_llm.head(MAX_CLUSTERS_TO_ANALYZE).copy()

print("Кластеров для анализа:", len(cluster_summary_for_llm))
display(cluster_summary_for_llm.head(30))

Кластеров для анализа: 297


,cluster_id,count,share
276,275,3966,0.022264
218,217,2821,0.015837
295,294,2776,0.015584
254,253,2442,0.013709
55,54,2212,0.012418
9,8,1789,0.010043
259,258,1765,0.009908
229,228,1708,0.009588
278,277,1538,0.008634
235,234,1456,0.008174


## 4. Список разрешенных классов

LLM будет не размечать отзывы, а сопоставлять кластер с классами, если это уместно.

In [10]:
ALLOWED_LABELS = [
    "Брак / дефект товара",
    "Низкое качество материала",
    "Проблема с размером / посадкой",
    "Несоответствие описанию",
    "Несоответствие ожиданиям / эффекту",
    "Проблема с комплектацией",
    "Проблема с упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Подделка / оригинальность",
    "Положительный отзыв",
    "Нейтральный / информационный отзыв",
    "Другая проблема",
]

## 5. Функции выбора примеров и подготовки prompt

In [11]:
def clean_review_for_prompt(text, max_review_chars=MAX_REVIEW_CHARS):
    text = str(text).strip()
    text = " ".join(text.split())

    if len(text) > max_review_chars:
        text = text[:max_review_chars] + "..."

    return text


def get_cluster_examples(
    df,
    cluster_id,
    n_examples=N_TOP_EXAMPLES,
    max_review_chars=MAX_REVIEW_CHARS,
    use_top_probability=True,
):
    cluster_df = df[df["cluster_id"] == cluster_id].copy()

    if cluster_df.empty:
        return []

    if use_top_probability and "cluster_probability" in cluster_df.columns:
        cluster_df = cluster_df.sort_values("cluster_probability", ascending=False)
    else:
        cluster_df = cluster_df.sample(
            n=min(len(cluster_df), n_examples),
            random_state=42,
        )

    cluster_df = cluster_df.head(n_examples)

    examples = []
    for _, row in cluster_df.iterrows():
        examples.append({
            "text": clean_review_for_prompt(row["text"], max_review_chars=max_review_chars),
            "cluster_probability": float(row.get("cluster_probability", 0.0)),
        })

    return examples

In [12]:
def build_cluster_analysis_prompt(cluster_id, cluster_size, cluster_share, examples):
    allowed_labels_text = "\n".join(f'- "{label}"' for label in ALLOWED_LABELS)

    examples_lines = []
    for i, ex in enumerate(examples, start=1):
        examples_lines.append(
            f"{i}. [prob={ex['cluster_probability']:.3f}] "
            f"{json.dumps(ex['text'], ensure_ascii=False)}"
        )
    examples_text = "\n".join(examples_lines)

    prompt = f"""
/no_think

Ты анализируешь кластер отзывов покупателей о товарах на маркетплейсе.

Кластер был получен автоматически через HDBSCAN по эмбеддингам отзывов.
Твоя задача — понять, какая общая особенность объединяет отзывы внутри кластера.

Важно:
- Анализируй только предоставленные примеры отзывов.
- Не додумывай факты, которых нет в примерах.
- Не утверждай, что весь кластер точно про эту тему, если примеры смешанные.
- Если кластер смешанный, прямо укажи это.
- Не размечай каждый отзыв отдельно.
- Нужно описать именно общую тему / особенность кластера.
- Можно сопоставить кластер с одним или несколькими классами из разрешенной схемы, если это уместно.

Разрешенные классы:
{allowed_labels_text}

Информация о кластере:
cluster_id = {cluster_id}
cluster_size = {cluster_size}
cluster_share = {cluster_share:.6f}

Примеры отзывов из кластера:
{examples_text}

Верни ответ строго в JSON формате:

{{
  "cluster_id": {cluster_id},
  "cluster_title": "короткое название кластера на русском",
  "cluster_feature": "главная особенность кластера одним-двумя предложениями",
  "main_theme": "главная тема кластера",
  "matched_labels": ["один или несколько классов из разрешенного списка, если подходят"],
  "is_mixed": false,
  "positive_or_negative": "positive / negative / mixed / neutral / unclear",
  "confidence": 0.0,
  "evidence": "краткое объяснение, какие признаки в примерах подтверждают вывод",
  "typical_phrases": ["типичная фраза 1", "типичная фраза 2", "типичная фраза 3"]
}}

Жесткие требования:
- Верни только один JSON-объект.
- Не возвращай markdown.
- Не возвращай текст вне JSON.
- matched_labels может содержать только классы из разрешенного списка.
- Если ни один класс не подходит, верни пустой список matched_labels.
- confidence — число от 0 до 1.
""".strip()

    return prompt


## 6. Функции вызова LLM и сохранения результата

In [13]:
def safe_json_loads(content):
    content = str(content).strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # На случай если модель все же вернула мусор вокруг JSON.
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(content[start:end + 1])
        raise


def normalize_cluster_analysis_response(content, cluster_id):
    data = safe_json_loads(content)

    matched_labels = data.get("matched_labels", [])
    if not isinstance(matched_labels, list):
        matched_labels = []

    matched_labels = [
        str(label) for label in matched_labels
        if str(label) in ALLOWED_LABELS
    ]

    try:
        confidence = float(data.get("confidence", 0.0))
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    typical_phrases = data.get("typical_phrases", [])
    if not isinstance(typical_phrases, list):
        typical_phrases = []
    typical_phrases = [str(x) for x in typical_phrases]

    return {
        "cluster_id": int(cluster_id),
        "cluster_title": str(data.get("cluster_title", "")),
        "cluster_feature": str(data.get("cluster_feature", "")),
        "main_theme": str(data.get("main_theme", "")),
        "matched_labels": matched_labels,
        "matched_labels_str": "; ".join(matched_labels),
        "is_mixed": bool(data.get("is_mixed", False)),
        "positive_or_negative": str(data.get("positive_or_negative", "unclear")),
        "confidence": confidence,
        "evidence": str(data.get("evidence", "")),
        "typical_phrases": typical_phrases,
        "typical_phrases_str": "; ".join(typical_phrases),
        "raw_response": str(content),
        "error": None,
    }

In [14]:
def analyze_cluster_with_llm(
    cluster_id,
    cluster_size,
    cluster_share,
    examples,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    max_retries=MAX_RETRIES,
):
    prompt = build_cluster_analysis_prompt(
        cluster_id=cluster_id,
        cluster_size=cluster_size,
        cluster_share=cluster_share,
        examples=examples,
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "/no_think\n"
                            "Ты ассистент для анализа кластеров отзывов покупателей. "
                            "Отвечай только валидным JSON-объектом. "
                            "Не добавляй markdown, рассуждения или текст вне JSON."
                        ),
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                temperature=temperature,
                response_format={"type": "json_object"},
            )

            content = response.choices[0].message.content
            return normalize_cluster_analysis_response(content, cluster_id)

        except Exception as e:
            error_text = str(e)

            # Явно выводим информацию о лимитах Groq/API
            if (
                "429" in error_text
                or "rate limit" in error_text.lower()
                or "tokens per day" in error_text.lower()
                or "TPD" in error_text
            ):
                print("\n" + "=" * 80)
                print("ДОСТИГНУТ ЛИМИТ API / RATE LIMIT")
                print(f"cluster_id: {cluster_id}")
                print(f"attempt: {attempt + 1}/{max_retries}")
                print("Ошибка:")
                print(error_text)
                print("=" * 80 + "\n")

            else:
                print("\n" + "=" * 80)
                print("ОШИБКА ПРИ АНАЛИЗЕ КЛАСТЕРА")
                print(f"cluster_id: {cluster_id}")
                print(f"attempt: {attempt + 1}/{max_retries}")
                print("Ошибка:")
                print(error_text)
                print("=" * 80 + "\n")
                raise RuntimeError(
                    "Остановлено: достигнут лимит API / rate limit. "
                    "Не продолжаю обработку, чтобы не записывать ошибки по всем кластерам."
                )

            if attempt == max_retries - 1:
                return {
                    "cluster_id": int(cluster_id),
                    "cluster_title": "",
                    "cluster_feature": "",
                    "main_theme": "",
                    "matched_labels": [],
                    "matched_labels_str": "",
                    "is_mixed": False,
                    "positive_or_negative": "unclear",
                    "confidence": 0.0,
                    "evidence": "",
                    "typical_phrases": [],
                    "typical_phrases_str": "",
                    "raw_response": "",
                    "error": error_text,
                }

            time.sleep(RETRY_BASE_SLEEP_SECONDS ** attempt)


In [15]:
def load_existing_cluster_analysis():
    if RESET_CLUSTER_ANALYSIS:
        for path in [
            CLUSTER_ANALYSIS_CSV_PATH,
            CLUSTER_ANALYSIS_PARQUET_PATH,
            CLUSTER_ANALYSIS_JSONL_PATH,
        ]:
            if path.exists():
                path.unlink()
        print("Старый LLM-анализ удален.")
        return [], set()

    if CLUSTER_ANALYSIS_CSV_PATH.exists():
        existing_df = pd.read_csv(CLUSTER_ANALYSIS_CSV_PATH)

        if "cluster_id" not in existing_df.columns:
            return [], set()

        done_cluster_ids = set(existing_df["cluster_id"].astype(int).tolist())
        print(f"Найден существующий анализ: {len(done_cluster_ids)} кластеров")
        return existing_df.to_dict("records"), done_cluster_ids

    return [], set()


def save_cluster_analysis(analysis_rows):
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

    analysis_df = pd.DataFrame(analysis_rows)

    # Для CSV списки дополнительно уже есть в *_str колонках.
    analysis_df.to_csv(CLUSTER_ANALYSIS_CSV_PATH, index=False)
    analysis_df.to_parquet(CLUSTER_ANALYSIS_PARQUET_PATH, index=False)

    with open(CLUSTER_ANALYSIS_JSONL_PATH, "w", encoding="utf-8") as f:
        for item in analysis_rows:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return analysis_df


## 7. Основной цикл анализа кластеров

Результат сохраняется после каждого кластера, поэтому если ноутбук остановится, его можно запустить повторно — он продолжит с места остановки.

In [16]:
analysis_rows, done_cluster_ids = load_existing_cluster_analysis()

clusters_to_process = cluster_summary_for_llm[
    ~cluster_summary_for_llm["cluster_id"].astype(int).isin(done_cluster_ids)
].copy()

print("Кластеров осталось обработать:", len(clusters_to_process))

pbar = tqdm(
    clusters_to_process.itertuples(index=False),
    total=len(clusters_to_process),
    desc="LLM cluster analysis",
)

for row in pbar:
    cluster_id = int(row.cluster_id)
    cluster_size = int(row.count)
    cluster_share = float(row.share)

    pbar.set_postfix({
        "cluster_id": cluster_id,
        "size": cluster_size,
    })

    examples = get_cluster_examples(
        result_df,
        cluster_id=cluster_id,
        n_examples=N_TOP_EXAMPLES,
        max_review_chars=MAX_REVIEW_CHARS,
        use_top_probability=USE_TOP_PROBABILITY_EXAMPLES,
    )

    analysis = analyze_cluster_with_llm(
        cluster_id=cluster_id,
        cluster_size=cluster_size,
        cluster_share=cluster_share,
        examples=examples,
    )

    analysis["cluster_size"] = cluster_size
    analysis["cluster_share"] = cluster_share
    analysis["n_examples_sent"] = len(examples)

    analysis_rows.append(analysis)

    # Сохраняем после каждого кластера, чтобы не потерять прогресс.
    analysis_df = save_cluster_analysis(analysis_rows)

    if REQUEST_SLEEP_SECONDS > 0:
        time.sleep(REQUEST_SLEEP_SECONDS)

print("Готово.")
print("CSV:", CLUSTER_ANALYSIS_CSV_PATH)
print("Parquet:", CLUSTER_ANALYSIS_PARQUET_PATH)
print("JSONL:", CLUSTER_ANALYSIS_JSONL_PATH)

Кластеров осталось обработать: 297


LLM cluster analysis:   0%|          | 0/297 [00:00<?, ?it/s]


ДОСТИГНУТ ЛИМИТ API / RATE LIMIT
cluster_id: 59
attempt: 1/3
Ошибка:
Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01ktcjgf01e1kazsmdxakz0det` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3336, Requested 2715. Please try again in 510ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


ДОСТИГНУТ ЛИМИТ API / RATE LIMIT
cluster_id: 112
attempt: 1/3
Ошибка:
Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01ktcjgf01e1kazsmdxakz0det` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499876, Requested 1937. Please try again in 5m13.2864s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


ДОСТИГНУТ ЛИМИТ API / RATE LIMIT
cluster_id: 112
attempt: 

KeyboardInterrupt: 

## 8. Просмотр результата анализа кластеров

In [ ]:
cluster_analysis_df = pd.read_csv(CLUSTER_ANALYSIS_CSV_PATH)

display_cols = [
    "cluster_id",
    "cluster_size",
    "cluster_title",
    "cluster_feature",
    "main_theme",
    "matched_labels_str",
    "is_mixed",
    "positive_or_negative",
    "confidence",
    "evidence",
    "error",
]

display_cols = [col for col in display_cols if col in cluster_analysis_df.columns]

display(
    cluster_analysis_df[display_cols]
    .sort_values("cluster_size", ascending=False)
    .head(50)
)

## 9. Объединение анализа с исходными отзывами

В итоговом файле у каждой строки будет `cluster_id` и описание соответствующего кластера.

In [ ]:
analysis_cols = [
    "cluster_id",
    "cluster_title",
    "cluster_feature",
    "main_theme",
    "matched_labels_str",
    "is_mixed",
    "positive_or_negative",
    "confidence",
]
analysis_cols = [col for col in analysis_cols if col in cluster_analysis_df.columns]

enriched_df = result_df.merge(
    cluster_analysis_df[analysis_cols],
    on="cluster_id",
    how="left",
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
enriched_df.to_parquet(ENRICHED_CLUSTERS_PATH, index=False)
enriched_df.to_csv(ENRICHED_CLUSTERS_CSV_PATH, index=False)

print("Сохранено:", ENRICHED_CLUSTERS_PATH)
print("Сохранено:", ENRICHED_CLUSTERS_CSV_PATH)
print("Размер:", enriched_df.shape)

display(enriched_df.head())

## 10. Примеры отзывов по выбранному кластеру

Можно вручную подставить `CLUSTER_ID_TO_VIEW` и посмотреть, насколько LLM-описание соответствует реальным отзывам.

In [ ]:
CLUSTER_ID_TO_VIEW = int(cluster_analysis_df.sort_values("cluster_size", ascending=False).iloc[0]["cluster_id"])

print("CLUSTER_ID_TO_VIEW:", CLUSTER_ID_TO_VIEW)

display(
    result_df[result_df["cluster_id"] == CLUSTER_ID_TO_VIEW]
    .sort_values("cluster_probability", ascending=False)
    [["text", "cluster_probability"]]
    .head(30)
)

display(
    cluster_analysis_df[cluster_analysis_df["cluster_id"] == CLUSTER_ID_TO_VIEW][display_cols]
)